# Demand-to-Replenishment Planning Engine

This notebook runs the repository pipeline on the UCI Online Retail transaction dataset, then reviews forecast accuracy, demand profiles, and illustrative inventory policies. Sales are a proxy for demand; costs, inventory, and lead times are scenario assumptions. See [README.md](README.md) for definitions and limits.

## Setup

From the repository root, run `python -m pip install -r requirements.txt` and `python download_data.py`. The raw spreadsheet is intentionally not committed.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from analysis import run

root = Path.cwd()
if not (root / "analysis.py").exists():
    raise RuntimeError("Open this notebook from the repository root so analysis.py is importable.")
results = run(root / "data" / "Online Retail.xlsx", root / "results")
import runpy
runpy.run_path("report.py")

## Data audit and item selection

Rows are filtered to positive, non-cancelled merchandise-like sales dated January through November 2011. The 20 SKUs with the greatest January–September selling revenue are selected before the October–November holdout. ABC classes are based on selling revenue, not inventory value.

In [ ]:
display(pd.Series(results["audit"], name="value").to_frame())
display(results["abc"].head(20))

## Fixed-method holdout forecasts

Each method predicts the next day using earlier observations only. WAPE is pooled across the 20 items and 61-day holdout. Croston SBA is included as an intermittent-demand benchmark; its performance here should not be generalized beyond these selected items.

In [ ]:
display(results["accuracy"].sort_values("wape"))
display(Image(filename=str(root / "results" / "forecast_comparison.png")))

## Rolling-origin validation and method selection

Four origins (May through August) use 28-day rolling one-step-ahead windows. The selected method for each SKU minimizes its pooled validation WAPE. Selection is then checked once on October–November; it is not assumed to improve accuracy.

In [ ]:
display(results["backtest_summary"].sort_values("wape"))
display(results["selected_methods"].sort_values("sku"))
display(results["selected_accuracy"])
display(Image(filename=str(root / "results" / "rolling_backtest_comparison.png")))

## Demand classification

ADI is training days divided by positive-demand days. CV² is the squared coefficient of variation of positive daily units. Thresholds of 1.32 and 0.49 label smooth, intermittent, erratic, or lumpy profiles.

In [ ]:
display(results["profiles"].sort_values(["demand_profile", "sku"]))
display(Image(filename=str(root / "results" / "demand_profile_mix.png")))

## Replenishment scenarios

The normal-theory buffer uses an assumed lead time and independent daily demand. The simulation warms inventory through September and reports October–November outcomes. A higher simulated fill rate uses more inventory in this scenario.

In [ ]:
display(results["summary"])
display(results["sensitivity"])
display(Image(filename=str(root / "results" / "policy_tradeoff.png")))
display(Image(filename=str(root / "results" / "lead_time_sensitivity.png")))

## Reorder-point optimization

The model tests empirical lead-time-demand quantiles and chooses the lowest assumed relevant cost on August validation. It holds that reorder point fixed for the October–November evaluation. These costs depend on stated assumptions and are not observed retailer costs.

In [ ]:
display(results["validation_candidates"].sort_values(["sku", "total_relevant_cost_gbp"]))
display(results["optimized_policy"])
display(Image(filename=str(root / "results" / "optimized_policy_tradeoff.png")))

## Interpretation

The source data does not reveal stock availability or lost demand. Replenishment metrics therefore measure the simulated fill of recorded sales under assumed policies. A live recommendation would require inventory, supplier, and cost history and validation on a later period.